## Set up Asyncio

In [8]:
import nest_asyncio

nest_asyncio.apply()

## Set up the Qdrant vector database

In [9]:
import qdrant_client

collection_name="chat_with_docs"

client = qdrant_client.QdrantClient(
    host="localhost",
    port=6333
)

## Read the documents

In [10]:
from llama_index.core import SimpleDirectoryReader

input_dir_path = './docs'

loader = SimpleDirectoryReader(
            input_dir = input_dir_path,
            required_exts=[".pdf"],
            recursive=True
        )
docs = loader.load_data()

In [11]:
type(docs), len(docs)

(list, 32)

## A function to index data

In [12]:
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, StorageContext

def create_index(documents):
    vector_store = QdrantVectorStore(client=client, collection_name=collection_name)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
    )
    return index

## Load the embedding model and index data

In [13]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-en-v1.5", trust_remote_code=True)
Settings.embed_model = embed_model

index = create_index(docs)

## Set up the LLM

In [15]:
import os 

OPENROUTER_KEY = os.environ["OPENROUTER_API_KEY"]

In [20]:
# from llama_index.llms.ollama import Ollama

# llm=Ollama(model="llama3.2:1b", request_timeout=120.0)

# Settings.llm = llm

from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

llm = OpenAI(
    model="gpt-4o-mini",
    api_base="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_KEY,
    temperature=0,
)

Settings.llm = llm

## Define the prompt template

In [21]:
from llama_index.core import PromptTemplate

qa_prompt_tmpl_str = (
"Context information is below.\n"
"---------------------\n"
"{context_str}\n"
"---------------------\n"
"Given the context information above I want you to think step by step to answer the query in a crisp manner, incase case you don't know the answer say 'I don't know!'.\n"
"Query: {query_str}\n"
"Answer: "
)

qa_prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)

## Reranking

In [22]:
from llama_index.core.postprocessor import SentenceTransformerRerank

rerank = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-2-v2", 
    top_n=3
)

## Query the document

In [24]:
query_engine = index.as_query_engine(similarity_top_k=10, node_postprocessors=[rerank])

query_engine.update_prompts(
    {"response_synthesizer:text_qa_template": qa_prompt_tmpl}
)

response = query_engine.query("What exactly is DSPy?")

## Print response

In [25]:
from IPython.display import Markdown, display

display(Markdown(str(response)))

DSPy is a programming framework designed to abstract and optimize the process of prompting language models (LMs) for natural language processing tasks. It utilizes natural language signatures, which are declarative specifications that define the input and output fields for text transformations, allowing users to specify what a function should do without detailing how to prompt the LM. DSPy includes modules like Predict, ChainOfThought, and others that facilitate the creation of parameterized and templated functions, enabling the development of efficient, self-improving NLP systems. The framework aims to enhance the performance of LMs by compiling these signatures into optimized prompts and reducing reliance on hand-crafted examples.